In [2]:
import polars as pl
df = pl.scan_parquet('./dataset/train.parquet/train.parquet')
train_labels = pl.read_csv('./dataset/train.parquet/train_labels.csv')
unique_customers = df.select("customer_ID").unique().collect()


In [2]:
feat_ids = df.select("customer_ID").unique().collect().get_column("customer_ID")
lab_ids = train_labels.get_column("customer_ID").unique()

missing_labels = feat_ids.filter(~feat_ids.is_in(lab_ids))
missing_features = lab_ids.filter(~lab_ids.is_in(feat_ids))

print(f"IDs in features without labels: {len(missing_labels)}")
print(f"IDs in labels without features: {len(missing_features)}")

if len(missing_labels) == 0 and len(missing_features) == 0:
    print("The customer sets are identical.")

C:\Users\HP\AppData\Local\Temp\ipykernel_23896\2131317591.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  missing_labels = feat_ids.filter(~feat_ids.is_in(lab_ids))


IDs in features without labels: 0
IDs in labels without features: 0
The customer sets are identical.


C:\Users\HP\AppData\Local\Temp\ipykernel_23896\2131317591.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  missing_features = lab_ids.filter(~lab_ids.is_in(feat_ids))


In [3]:
print(type(df))
print(type(train_labels))

<class 'polars.lazyframe.frame.LazyFrame'>
<class 'polars.dataframe.frame.DataFrame'>


In [4]:
merged_data = (
    df.lazy().join(train_labels.lazy() if isinstance(train_labels , pl.DataFrame) else train_labels , on = "customer_ID" , how = "left").collect(streaming = True)
)

C:\Users\HP\AppData\Local\Temp\ipykernel_16340\527977169.py:2: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df.lazy().join(train_labels.lazy() if isinstance(train_labels , pl.DataFrame) else train_labels , on = "customer_ID" , how = "left").collect(streaming = True)


In [4]:
for col, dtype in merged_data.schema.items():
    if dtype == pl.String: 
        print(f"Column: {col} | Type: {dtype}")

Column: customer_ID | Type: String
Column: S_2 | Type: String


In [ ]:
numeric_cols = merged_data.select(pl.col(pl.Float32, pl.Float64, pl.Int32, pl.Int16, pl.Int8)).columns
numeric_cols = [c for c in numeric_cols if c not in ['customer_ID', 'target']]

cleaned_data = merged_data.with_columns([
    pl.col(numeric_cols).fill_null(-999)
])


cleaned_data = cleaned_data.with_columns([
    pl.col("S_2").fill_null(pl.date(1900, 1, 1))
])

# print(cleaned_data.collect_schema().names())

['customer_ID', 'S_2', 'P_2', 'D_39', 'B_1', 'B_2', 'R_1', 'S_3', 'D_41', 'B_3', 'D_42', 'D_43', 'D_44', 'B_4', 'D_45', 'B_5', 'R_2', 'D_46', 'D_47', 'D_48', 'D_49', 'B_6', 'B_7', 'B_8', 'D_50', 'D_51', 'B_9', 'R_3', 'D_52', 'P_3', 'B_10', 'D_53', 'S_5', 'B_11', 'S_6', 'D_54', 'R_4', 'S_7', 'B_12', 'S_8', 'D_55', 'D_56', 'B_13', 'R_5', 'D_58', 'S_9', 'B_14', 'D_59', 'D_60', 'D_61', 'B_15', 'S_11', 'D_62', 'D_63', 'D_64', 'D_65', 'B_16', 'B_17', 'B_18', 'B_19', 'D_66', 'B_20', 'D_68', 'S_12', 'R_6', 'S_13', 'B_21', 'D_69', 'B_22', 'D_70', 'D_71', 'D_72', 'S_15', 'B_23', 'D_73', 'P_4', 'D_74', 'D_75', 'D_76', 'B_24', 'R_7', 'D_77', 'B_25', 'B_26', 'D_78', 'D_79', 'R_8', 'R_9', 'S_16', 'D_80', 'R_10', 'R_11', 'B_27', 'D_81', 'D_82', 'S_17', 'R_12', 'B_28', 'R_13', 'D_83', 'R_14', 'R_15', 'D_84', 'R_16', 'B_29', 'B_30', 'S_18', 'D_86', 'D_87', 'R_17', 'R_18', 'D_88', 'B_31', 'S_19', 'R_19', 'B_32', 'S_20', 'R_20', 'R_21', 'B_33', 'D_89', 'R_22', 'R_23', 'D_91', 'D_92', 'D_93', 'D_94', 'R_2

In [7]:
all_features = [c for c in cleaned_data.columns if c not in ["customer_ID","target","S_2"]]
agg_exprs = [
    *[pl.col(c).mean().alias(f"{c}_mean") for c in all_features],
    *[pl.col(c).last().alias(f"{c}_last") for c in all_features],
    *[(pl.col(c).mean() - pl.col(c).last()).alias(f"{c}_delta") for c in all_features],
]


features_profiles = (
    cleaned_data.lazy()
    .group_by(["customer_ID","target"])
    .agg(agg_exprs)
    .collect(streaming = True)
)

print(features_profiles.collect_schema().names())
# features_profiles.write_parquet("final_data.parquet")

C:\Users\HP\AppData\Local\Temp\ipykernel_16340\125001431.py:13: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming = True)


['customer_ID', 'target', 'P_2_mean', 'D_39_mean', 'B_1_mean', 'B_2_mean', 'R_1_mean', 'S_3_mean', 'D_41_mean', 'B_3_mean', 'D_42_mean', 'D_43_mean', 'D_44_mean', 'B_4_mean', 'D_45_mean', 'B_5_mean', 'R_2_mean', 'D_46_mean', 'D_47_mean', 'D_48_mean', 'D_49_mean', 'B_6_mean', 'B_7_mean', 'B_8_mean', 'D_50_mean', 'D_51_mean', 'B_9_mean', 'R_3_mean', 'D_52_mean', 'P_3_mean', 'B_10_mean', 'D_53_mean', 'S_5_mean', 'B_11_mean', 'S_6_mean', 'D_54_mean', 'R_4_mean', 'S_7_mean', 'B_12_mean', 'S_8_mean', 'D_55_mean', 'D_56_mean', 'B_13_mean', 'R_5_mean', 'D_58_mean', 'S_9_mean', 'B_14_mean', 'D_59_mean', 'D_60_mean', 'D_61_mean', 'B_15_mean', 'S_11_mean', 'D_62_mean', 'D_63_mean', 'D_64_mean', 'D_65_mean', 'B_16_mean', 'B_17_mean', 'B_18_mean', 'B_19_mean', 'D_66_mean', 'B_20_mean', 'D_68_mean', 'S_12_mean', 'R_6_mean', 'S_13_mean', 'B_21_mean', 'D_69_mean', 'B_22_mean', 'D_70_mean', 'D_71_mean', 'D_72_mean', 'S_15_mean', 'B_23_mean', 'D_73_mean', 'P_4_mean', 'D_74_mean', 'D_75_mean', 'D_76_mean